# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [15]:
# imports
import os
from textwrap import dedent
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [16]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2:1b'

In [17]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

API key looks good so far


In [18]:
system_prompt = """
You are a specialized "Code Explainer" AI, designed to assist a developer transitioning from web development into AI and Machine Learning. Your primary function is to perform deep-dive technical audits and logic explanations for Python and AI-related codebases.

### Core Identity & Logic
* **The "Why" Over the "What":** Do not just describe what a line of code does; explain the architectural reason why it was written that way and the performance implications of the choice.
* **Stable & Scalable:** Always frame your explanations through the lens of production-ready, scalable code suitable for a freelancer or independent developer.
* **Contextual Awareness:** Look for patterns such as generator delegation, list comprehensions, and safe dictionary access, and explain how these patterns contribute to cleaner code.

### Guidelines for Explanations
* **Depth and Clarity:** Provide long, detailed, and comprehensive responses. Every concept should be explained with enough depth to be easy to understand without sacrificing technical accuracy.
* **No Analogies:** Strictly avoid using analogies to explain technical concepts unless specifically requested. Rely on direct technical mechanics.
* **Step-by-Step Methodology:** Break down complex code blocks or algorithms into logical, sequential parts. Pause or use headings to separate the "Input," "Transformation," and "Output" phases of the code.
* **Latest Documentation:** Ensure all explanations align with the latest stable versions of libraries (e.g., OpenAI API v1.0+, LangChain, PyTorch).
* **Technical Jargon:** Use professional terms (e.g., "idempotency," "asynchronous execution," "generator delegation") and provide clear definitions for each.

### Formatting Requirements
* **Markdown Hierarchy:** Use clear Markdown structure with headings (##, ###) and horizontal rules (---).
* **Syntax Highlighting:** All code must be enclosed in triple backticks with the language identifier (e.g., ```python).
* **Visual Scannability:** Use bolding for function names/variables and use tables when comparing different coding approaches (e.g., `for` loops vs. `comprehensions`).

### Trigger Response
* When asked "Please explain what this code does and why:", perform a line-by-line breakdown followed by a "Rationale" section explaining the benefit of that specific syntax.
"""


def craft_user_prompt(code_block):
    # Remove leading/trailing whitespace and dedent the code
    clean_code = dedent(code_block).strip()
    
    user_prompt = f"""Please explain what this code does and why:

```python
{clean_code}
```"""
    
    return user_prompt

In [19]:
code = """
    import functools, time

    def cache(f, c={}):
        @functools.wraps(f)
        def w(*a):
            if a not in c: c[a] = (f(*a), time.time())
            if len(c) > 3: del c[min(c, key=lambda k: c[k][1])]
            return c[a][0]
        return w

"""

craft_user_prompt(code)

'Please explain what this code does and why:\n\n```python\nimport functools, time\n\ndef cache(f, c={}):\n    @functools.wraps(f)\n    def w(*a):\n        if a not in c: c[a] = (f(*a), time.time())\n        if len(c) > 3: del c[min(c, key=lambda k: c[k][1])]\n        return c[a][0]\n    return w\n```'

In [22]:
# Get the model to answer, with streaming
def stream_explanation(code_block, provider ,model):
    stream = provider.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": craft_user_prompt(code_block)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_explanation(code, openai, MODEL_GPT)

## Line-by-line explanation (What this code does and why)

### Line-by-line breakdown

1) ```python
   import functools, time
   ```
   - Brings in the standard library modules used later:
     - functools.wraps to preserve the wrapped function's metadata.
     - time.time() to timestamp cache insertions for eviction.

2) ```python
   def cache(f, c={}):
   ```
   - Defines a decorator factory named `cache`.
   - It takes a function `f` to wrap.
   - The second parameter `c` has a default value of an empty dict. That default dict is mutable and mutable defaults persist across calls, which has important behavioral implications (see “Rationale” and “Potential issues” below).
   - The intent is to create a closure over `c` that serves as a cache for the wrapped function.

3) ```python
   @functools.wraps(f)
   ```
   - Applies a decorator to the inner wrapper to preserve the wrapped function’s metadata (name, docstring, etc.), which is important for debuggability and tooling.

4) ```python
   def w(*a):
   ```
   - Defines the actual wrapper function `w` that will replace `f`.
   - It accepts any number of positional arguments via `*a`.
   - Important note: There is no `**kwargs` parameter, so keyword arguments are not supported by this wrapper.

5) ```python
   if a not in c: c[a] = (f(*a), time.time())
   ```
   - Checks if the argument tuple `a` is already a cache key in `c`.
   - If not cached, computes `f(*a)` and stores a tuple `(result, timestamp)` in the cache under key `a`.
   - The timestamp is `time.time()` at the moment of caching, which will be used for eviction.

6) ```python
   if len(c) > 3: del c[min(c, key=lambda k: c[k][1])]
   ```
   - Evicts an entry when the cache grows beyond 3 items.
   - It finds the key with the smallest timestamp `c[k][1]` (i.e., the oldest insertion time) and deletes that entry.
   - This implements a simple fixed-capacity cache with a FIFO-like eviction policy (oldest insertion is removed).

7) ```python
   return c[a][0]
   ```
   - Returns the cached result for the current arguments, i.e., the first element of the stored tuple `(result, timestamp)`.

8) ```python
   return w
   ```
   - Returns the wrapper function `w`, completing the decorator.

---

## Rationale (why each design choice was made)

- Closure over `c`:
  - The cache lives in the function’s closure so every call to the decorated function can reuse the stored results without re-creating the cache. This is the core mechanism that makes the function memoized.

- Mutable default argument for `c`:
  - The use of a mutable default argument is what actually carries the cache across calls. In a production setting, this is a common pitfall: the same dict is shared across all uses of `cache` that don’t pass an explicit `c`. This enables the cache to be shared across different decorated functions, which is almost certainly unintended and can lead to incorrect results if different functions are wrapped with the same decorator instance.

- `functools.wraps(f)`:
  - Preserves the wrapped function’s identity (name, doc, signature) which is essential for debugging, introspection, and tooling in production code.

- Positional-args-only wrapper (`*a`, no `**kwargs`) and hashing:
  - The cache key is the tuple of positional arguments `a`. This makes the wrapper simple and fast but means:
    - Keyword arguments are not supported (they’ll trigger a TypeError if passed).
    - Arguments must be hashable (e.g., cannot include lists or dicts directly, since they would make `a` unhashable).

- Simple eviction policy (FIFO by insertion time):
  - When the cache exceeds size 3, the oldest entry (earliest timestamp) is removed.
  - This keeps memory usage bounded and ensures the cache remains small, but it’s not a true LRU (least-recently-used) cache since timestamps aren’t updated on access.

- Storage of `(result, timestamp)`:
  - Keeps both the computed value and the time it was cached, enabling straightforward eviction decisions based on age.

---

## Potential issues and recommended fixes (production-oriented)

- Mutable default argument pitfall
  - Issue: The default `c={}` is shared across all decorated functions that use the default, causing cross-function cache interference.
  - Fix: Create a per-function cache inside the decorator factory, not via a mutable default. Example approach:

    ```python
    import functools, time

    def cache(maxsize=3):
        def decorator(f):
            c = {}

            @functools.wraps(f)
            def w(*a, **kwargs):
                key = (a, tuple(sorted(kwargs.items()))) if kwargs else (a,)
                if key not in c:
                    c[key] = (f(*a, **kwargs), time.time())
                if len(c) > maxsize:
                    del c[min(c, key=lambda k: c[k][1])]
                return c[key][0]
            return w
        return decorator
    ```

- Support for keyword arguments
  - Issue: The current wrapper does not accept or forward `**kwargs`.
  - Fix: Include `**kwargs` in the wrapper and construct a cache key that accounts for both positional and keyword arguments (as shown in the improved snippet above). This makes the decorator more robust and usable with real-world functions.

- True LRU vs FIFO eviction
  - Issue: The current eviction removes the oldest insertion, not the least recently used.
  - Fix (optional): Implement an LRU policy using an ordered structure (e.g., collections.OrderedDict or a dedicated cache library like cachetools). For example, with OrderedDict you can move the accessed key to the end on every hit and evict from the front.

- Thread-safety
  - Issue: The dict `c` is not protected for concurrent access.
  - Fix: Add a threading.Lock around reads/writes if the decorated function will be used from multiple threads, or use a thread-safe cache structure.

- Memory growth risk and saturation
  - If the decorator is misused (e.g., many unique calls with distinct argument tuples), memory can grow.
  - Fix: Consider using a more robust cache with configurable eviction policy, or integrate with a library like cachetools.

---

## Minimal usage example (with the original behavior)

Note: The original code has the mutable-default pitfall. This example demonstrates the exact behavior but is not recommended for production.

```python
@cache
def add(a, b):
    print(f"Computing {a} + {b}")
    return a + b

print(add(1, 2))  # Computes and caches
print(add(1, 2))  # Returns cached value
```

This will print the computation message only on the first call for the arguments (1, 2) and reuse the cached result on the second.

---

## Summary

- The code implements a simple memoization decorator with a tiny fixed-size cache (3 entries) using a FIFO eviction policy based on insertion time.
- It relies on a mutable default argument for the cache, which leads to cross-function cache leakage when used as a decorator on multiple functions—a critical pitfall for production code.
- It supports only positional arguments and does not handle keywords, limiting real-world applicability.
- For production-quality code, refactor to create per-function caches, add keyword argument support, consider a true LRU policy, and address thread-safety.

In [26]:
# Get Llama 3.2 to answer
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
stream_explanation(code, ollama, MODEL_LLAMA)

**Code Breakdown**

This code defines a caching function using Python's `functools` module and creates a decorator called `cache`. The main purpose of this caching function is to store the results of expensive function calls (functions that perform repeated calculations) so they don't need to be recalculated every time.

Here's a step-by-step breakdown:

1. **Import Modules**: 
   - `functools` provides helper functions for functional programming.
   - `time` module allows us to access and manipulate the current time.

2. **Define the cache decorator**:
   The `cache` function takes another function `f` as an argument and returns a new "wrapper" function `w`. This wrapper will apply the caching mechanism to the original function `f`.

3. **Wrapper Function (wr)**:
   - The wrapper function has two main tasks:
     - It checks if the cached result is already stored in the dictionary `c[a]`.
     - If not, it calculates the result using `f(*a)` and stores everything in a tuple `(result, time.time())` where `time.time()` gets the current timestamp.
     - Finally, it checks if the cache size (`len(c)`) exceeds three. In this case, it removes the oldest cached item from the list.

4. **Decorating Function**:
   This is where the magic happens:
   - The original function `f` and its name along with the cache key `a` are stored in a dictionary named `c`, by default.
   - If any values are modified, they will be written to `c`.

5. **Return Statement**:
   The wrapper function (`wr`) should return the cached result or calculate it if it's not yet stored.

6. **Calling Decorator**:
   This is done using an example of a function `g` with unknown caching behavior (we'll show later), where some operations are performed and results are printed to verify usage.
7. **Example Usage**:

```python
import time

@gcache
def g(n):
    start_time = time.time()
    result = sum(range(n))
    end_time = time.time()
    print(f"Result = {result}, Time = {(end_time - start_time) * 1000} ms")
```
The output would look something like this:

```
Result = 21, Time = (1.0101720000000096s)
Result = 22, Time = (2.0101720000000043s)
Result = 23, Time = (3.0101720000000019s)

```

**Explanation**

The main benefit of this caching function is to avoid re-calculating values if they can be computed just once and then stored in a dictionary for future reference.

However, the use of dictionaries here is a bit unusual because lists aren't ideal data structures here. Here's why:

- `functools.wraps(f)` preserves instance-specific attributes (like `__dict__`) that we can access like regular variables.
- Caching this way assumes all our functions and their arguments are small enough to be stored in lists as tuples, which they often won't be.
- But caching a result of an operation doesn't necessarily mean the values should be cached.

In contrast, lists or tuples would allow for much more efficient caching because they're inherently good at storing large blocks of data. Also, here's how you can manually remove old cache entries without using `functools.wraps(f):
```python
def cache_f(c={}, n=1000):
    if len(c) >= 3: del c[min(c, key=lambda k: c[k][1])]
```
This way we're making sure to drop any older items before adding the new item.